# ระดับ 3c — ประเมินโมเดลที่เทรนเอง (cross-dataset) เทียบกับ paper

**Paper:** Guo et al., *Revisiting Face Forgery Detection: From Facial Representation to Forgery Detection*, IEEE TPAMI 2026

**เป้าหมาย:** เอา checkpoint จากระดับ 3b (เทรนบน FF++ c23 เท่านั้น) ไปวัดบน **Celeb-DF v2** ซึ่งเป็น unseen dataset
แล้วเทียบ 3 ทาง: (1) ตัวเลขใน paper, (2) checkpoint ของผู้เขียน, (3) โมเดลที่เราเทรนเอง

| | FF++ c23 | Celeb-DF |
|---|---|---|
| paper (Ours) | 99.36 | 90.46 |
| paper (BEiT v2′ ไม่มี CFB) | 99.14 | 89.18 |
| checkpoint ผู้เขียน (คำนวณใหม่จาก csv ที่แนบ) | ในหัวข้อ 6 | ในหัวข้อ 6 |
| **โมเดลของเรา** | จากระดับ 3b | **หัวข้อ 5** |

**Protocol (ตาม repo):** 8 frame/วิดีโอ → RetinaFace → align → crop 224×224 → main branch → softmax →
เฉลี่ย 8 frame = score ของวิดีโอ → **video-level AUC**

**ก่อนรัน:** GPU · Internet = On · Add Input:
1. output ของระดับ 3b (มี `cfb_run/best.pth`)
2. Celeb-DF v2 test — dataset `celebdf_v2_test518` ที่สร้างจาก `prepare_celebdf_test.py` (ถ้าไม่มีจะดาวน์โหลดจาก Drive)
3. *(ไม่บังคับ)* output ของระดับ 2 ที่มี `cdf_crops` อยู่แล้ว — จะข้ามขั้น crop ไปเลย

## 0. ตั้งค่า

In [ ]:
import os, sys, re, json, time, glob, random, shutil, zipfile, hashlib, subprocess, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
T_START = time.time()

ON_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR  = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("FFD_WORK", "./work")).resolve()
TMP_DIR   = Path("/tmp/ffd") if ON_KAGGLE else WORK_DIR / "tmp"
INPUT_DIR = Path("/kaggle/input") if ON_KAGGLE else Path(os.environ.get("FFD_INPUT", str(WORK_DIR / "input")))
REPO_DIR  = TMP_DIR / "FFDBackbone"
CROP_DIR  = WORK_DIR / "cdf_crops"
OUT_DIR   = WORK_DIR / "results_level3c"
for d in [TMP_DIR, CROP_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/zhenglab/FFDBackbone.git"

# --- checkpoint ของเรา (จากระดับ 3b) ---
OUR_CKPT = None            # None = ค้นหา cfb_run/best.pth ใน /kaggle/input อัตโนมัติ

# --- Celeb-DF v2 ---
CDF_ROOT = None            # None = ค้นหา List_of_testing_videos.txt ใน /kaggle/input
CDF_GDRIVE_ID = "1iLx76wsbi9itnkxSqz9BVBl4ZvnbIazj"
CDF_ZIP_BYTES = 9_952_957_051

# --- preprocessing / inference (ค่าตาม repo) ---
N_FRAMES, IMG_SIZE = 8, 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
DET_THRES, MAX_SHIFT = 0.5, 10
BATCH_SIZE, NUM_WORKERS, USE_AMP = 64, 2, True

# --- analysis ---
UTOM_LAMBDA, UTOM_EPS = 0.02, 1e-7
N_BOOT, SEED = 1000, 42
PAPER = {"FaceForensics_c23": 99.36, "Celeb-DF": 90.46, "DFDC": 84.90, "FFIW": 90.97}

random.seed(SEED)
print("ON_KAGGLE:", ON_KAGGLE)

## 1. ติดตั้ง + 2. Clone repo + patch

In [ ]:
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs], check=False)
pip_install("gdown")
try: import timm
except ImportError: pip_install("timm"); import timm

import numpy as np, pandas as pd, cv2, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print({"torch": torch.__version__, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"})

def find_repo_in_input():
    """หา FFDBackbone ที่ Add Input ไว้ (โหมดออฟไลน์) — ค้นแค่ 2 ชั้นแรก ไม่ไล่ทั้ง dataset"""
    if not INPUT_DIR.exists():
        return None
    marker = Path("models") / "lib" / "BEiT_v2" / "modeling_finetune.py"
    for d in sorted(p for p in INPUT_DIR.iterdir() if p.is_dir()):
        cands = [d] + sorted(p for p in d.iterdir() if p.is_dir())[:50]
        for c in cands:
            if (c / marker).exists():
                return c
    return None

def get_repo():
    """1) ที่ clone ไว้แล้ว  2) จาก /kaggle/input (ออฟไลน์)  3) git clone"""
    if (REPO_DIR / "models" / "lib" / "BEiT_v2" / "modeling_finetune.py").exists():
        return "cached"
    src = find_repo_in_input()
    if src is not None:
        print("พบ repo ใน Add Input (โหมดออฟไลน์):", src)
        shutil.copytree(src, REPO_DIR, dirs_exist_ok=True)
        return "input"
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(
            "clone repo ไม่สำเร็จ:\n  " + (r.stderr or "").strip() + "\n\n"
            "ถ้าขึ้น 'Could not resolve host' แปลว่า Internet ปิดอยู่ — แก้ได้ 2 ทาง\n"
            "  (ก) เปิดเน็ต: แผงขวาของ Kaggle -> Notebook options -> Internet = On\n"
            "      (ต้องยืนยันเบอร์โทรที่ kaggle.com/settings ก่อน ถ้ายังไม่เคยทำ)\n"
            "  (ข) โหมดออฟไลน์: ดาวน์โหลด zip ของ repo จาก\n"
            "      https://github.com/zhenglab/FFDBackbone/archive/refs/heads/main.zip\n"
            "      -> อัปเป็น Kaggle Dataset -> Add Input  (cell นี้จะหาเจอเอง)")
    return "clone"

REPO_SOURCE = get_repo()
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip() or "(จาก Add Input — ไม่มี .git)"

def patch_repo(repo: Path):
    n = 0
    for py in list((repo / "face_process").rglob("*.py")) + list((repo / "models" / "lib" / "BEiT_v2").rglob("*.py")):
        src = py.read_text(encoding="utf-8", errors="ignore")
        new = re.sub(r"np\.float\b", "np.float64", src)
        new = re.sub(r"np\.int\b", "np.int64", new)
        new = re.sub(r"np\.bool\b", "np.bool_", new)
        new = new.replace("np.reshape(b, newshape=(1, 2))", "np.reshape(b, (1, 2))")
        if py.name == "alignment.py":
            if "import torch.utils.model_zoo" not in new:
                new = "import torch.utils.model_zoo\n" + new
            new = new.replace("model_zoo.load_url(url)", "model_zoo.load_url(url, map_location='cpu')")
        if new != src:
            py.write_text(new, encoding="utf-8"); n += 1
    return n

print("repo commit:", REPO_COMMIT, "| patched:", patch_repo(REPO_DIR))
if str(REPO_DIR) not in sys.path: sys.path.insert(0, str(REPO_DIR))

import importlib.util
def load_module_from_file(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(spec); sys.modules[name] = m
    spec.loader.exec_module(m); return m

modeling_finetune = load_module_from_file("beitv2_modeling_finetune",
                                          REPO_DIR / "models" / "lib" / "BEiT_v2" / "modeling_finetune.py")
from face_process.lib.ct.detection import FaceDetector
from face_process import face_utils

## 3. โหลด checkpoint ของเรา

โครงสร้างเดียวกับ `DDBF_BEiT_v2` ของ repo — inference ใช้ **main branch** (`backbone_model`) ตาม paper
และคำนวณ `fused` ไว้เปรียบเทียบด้วย

⚠️ ตรวจว่า **ทุก key โหลดเข้าครบ** (ไม่ปล่อยให้ `strict=False` กลืน error เงียบ ๆ)

In [ ]:
BEIT_CONFIG = dict(drop_path_rate=0.1, use_mean_pooling=False, init_values=0.1, qkv_bias=True,
                   use_abs_pos_emb=False, use_rel_pos_bias=True, use_shared_rel_pos_bias=False)

def make_beit():
    m = modeling_finetune.beit_base_patch16_224(**BEIT_CONFIG)
    m.head = nn.Identity()
    return m

def evidential_uncertainty(logits, K=2):
    alpha = torch.exp(torch.clamp(logits, -10, 10)) + 1
    return K / alpha.sum(dim=1, keepdim=True)

class CFBDetector(nn.Module):
    def __init__(self, dual: bool):
        super().__init__()
        self.dual = dual
        self.backbone_model = make_beit()
        if dual: self.backbone_model2 = make_beit()
        self.norm_for_cls = nn.LayerNorm(768)
        self.header = nn.Sequential(nn.Linear(768, 2))
    def classify(self, f): return self.header(self.norm_for_cls(f))
    def forward(self, x):
        f1 = self.backbone_model(x)
        out = {"main": self.classify(f1)}
        if self.dual:
            f2 = self.backbone_model2(x)
            l2 = self.classify(f2)
            u = torch.cat([evidential_uncertainty(out["main"].float()),
                           evidential_uncertainty(l2.float())], dim=-1)
            w = torch.softmax(-10 * u, dim=-1).to(f1.dtype)
            out["branch2"] = l2
            out["fused"] = self.classify(w[:, :1] * f1 + w[:, 1:] * f2)
        return out

def find_our_ckpt():
    if OUR_CKPT and Path(OUR_CKPT).exists(): return Path(OUR_CKPT)
    for pat in ["cfb_run/best.pth", "cfb_run/last.pth", "**/best.pth"]:
        hits = sorted(INPUT_DIR.rglob(pat)) if INPUT_DIR.exists() else []
        if hits: return max(hits, key=lambda p: p.stat().st_mtime)
    raise RuntimeError("ไม่พบ checkpoint จากระดับ 3b — Add Input เป็น output ของ notebook FFD_level3b")

ckpt_file = find_our_ckpt()
ck = torch.load(ckpt_file, map_location="cpu", weights_only=False)
sd = ck.get("state_dict", ck)
sd = {(k[7:] if k.startswith("module.") else k): v for k, v in sd.items() if torch.is_tensor(v)}
DUAL = any(k.startswith("backbone_model2.") for k in sd)
print(f"checkpoint: {ckpt_file}  ({ckpt_file.stat().st_size/1e6:.0f} MB)")
print("epoch:", ck.get("epoch"), "| val AUC ตอนบันทึก:", round(100*ck.get("val_auc", float('nan')), 2), "| dual:", DUAL)

model = CFBDetector(dual=DUAL)
expected = set(model.state_dict())
loadable = {k: v for k, v in sd.items() if k in expected}
missing = sorted(expected - set(loadable))
shape_bad = [k for k, v in loadable.items() if v.shape != model.state_dict()[k].shape]
print(f"keys: expected={len(expected)} loaded={len(loadable)} missing={len(missing)} "
      f"unexpected={len(set(sd)-expected)}")
if missing or shape_bad:
    raise RuntimeError(f"checkpoint ไม่ตรงกับโมเดล\nmissing[:10]={missing[:10]}\nshape={shape_bad[:5]}")
model.load_state_dict(loadable, strict=True)
model.eval().to(DEVICE)
print("โหลด weight ครบทุก key ✓")

## 4. Celeb-DF v2 — หาไฟล์ + ดึง frame + crop หน้า
เหมือนระดับ 2 ทุกขั้น · ถ้า Add Input เป็น output ของระดับ 2 ที่มี `cdf_crops` อยู่แล้ว จะข้ามขั้นนี้

⚠️ ใน `List_of_testing_videos.txt` **1 = real, 0 = fake** (กลับกับโมเดล) → แปลงเป็น `label = 1 − label_official`
เหมือน `get_data_list()` ของ repo (`FAKE_LABEL - int(line[0])`)

In [ ]:
detector = FaceDetector(0 if DEVICE.type == "cuda" else -1)

def valid_faces(faces, max_count=10, thres=DET_THRES):
    out = []
    for box, lm, score in faces[:max_count]:
        if score < thres: continue
        out.append((box.astype(np.float64), lm.astype(np.float64), float(score)))
    return out

def crop_face_test(img, bbox):
    H, W = img.shape[:2]
    (x0, y0), (x1, y1) = bbox
    w, h = x1 - x0, y1 - y0
    w0 = w1 = w / 4 * 0.5; h0 = h1 = h / 4 * 0.5
    return img[max(0, int(y0-h0)):min(H, int(y1+h1)+1), max(0, int(x0-w0)):min(W, int(x1+w1)+1)]

def align_and_crop(img_bgr, face):
    bbox, lm5, _ = face
    aligned, _, bb = face_utils.crop_aligned(img_bgr, lm5, landmarks_68=None, bboxes=bbox,
                                            aligned_image_size=IMG_SIZE)
    crop = crop_face_test(aligned, np.array([[bb[0], bb[1]], [bb[2], bb[3]]]))
    if crop.size == 0: return None
    return cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)

def box_iou(a, b):
    x0, y0 = max(a[0], b[0]), max(a[1], b[1]); x1, y1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0., x1-x0) * max(0., y1-y0)
    area = lambda r: max(0., r[2]-r[0]) * max(0., r[3]-r[1])
    return inter / (area(a) + area(b) - inter + 1e-9)

def pick_face(faces, prev):
    return (max(faces, key=lambda f: (f[0][2]-f[0][0])*(f[0][3]-f[0][1])) if prev is None
            else max(faces, key=lambda f: box_iou(f[0], prev)))

def sample_indices(n, k=N_FRAMES):
    idx = np.arange(n)
    return idx[::max(1, n // k)][:k] if n >= k else np.sort(np.resize(idx, k))

def process_video(path, out_dir):
    cap = cv2.VideoCapture(str(path))
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if n_total <= 0: cap.release(); return [], {"error": "no_frames"}
    out_dir.mkdir(parents=True, exist_ok=True)
    recs, prev = [], None
    for target in sample_indices(n_total):
        for shift in range(MAX_SHIFT + 1):
            fi = int(target) + shift
            if fi >= n_total: break
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ok, frame = cap.read()
            if not ok or frame is None: continue
            try: faces = valid_faces(detector.detect(frame))
            except Exception: continue
            if not faces: continue
            face = pick_face(faces, prev)
            crop = align_and_crop(frame, face)
            if crop is None: continue
            p = out_dir / f"{fi}.png"
            cv2.imwrite(str(p), crop)
            prev = face[0]
            recs.append({"crop_path": str(p), "frame_idx": fi, "shift": shift})
            break
    cap.release()
    return recs, {"n_total_frames": n_total}

In [ ]:
def find_cdf_root():
    if CDF_ROOT and (Path(CDF_ROOT) / "List_of_testing_videos.txt").exists(): return Path(CDF_ROOT)
    hits = sorted(INPUT_DIR.rglob("List_of_testing_videos.txt")) if INPUT_DIR.exists() else []
    if hits: return hits[0].parent
    prev = sorted((TMP_DIR / "cdf").rglob("List_of_testing_videos.txt")) if (TMP_DIR / "cdf").exists() else []
    if prev: return prev[0].parent
    import gdown
    zp = TMP_DIR / "Celeb-DF-v2.zip"
    if not (zp.exists() and zp.stat().st_size == CDF_ZIP_BYTES):
        print(f"ดาวน์โหลด Celeb-DF-v2.zip (9.95 GB) · ว่าง {shutil.disk_usage(TMP_DIR).free/1e9:.0f} GB")
        gdown.download(id=CDF_GDRIVE_ID, output=str(zp), quiet=False, resume=True)
    dst = TMP_DIR / "cdf"
    with zipfile.ZipFile(zp) as z:
        names = z.namelist()
        lst = [n for n in names if n.endswith("List_of_testing_videos.txt")][0]
        prefix = lst[: -len("List_of_testing_videos.txt")]
        z.extract(lst, dst)
        wanted = [prefix + l.split()[1] for l in z.read(lst).decode().split("\n") if len(l.split()) == 2]
        have = set(names)
        for n in tqdm([w for w in wanted if w in have], desc="extract test videos"):
            if not (dst / n).exists(): z.extract(n, dst)
    return dst / prefix

cdf_root = find_cdf_root()
rows = []
for line in (cdf_root / "List_of_testing_videos.txt").read_text().split("\n"):
    parts = line.split()
    if len(parts) == 2:
        rel = parts[1].replace("\\", "/")
        rows.append({"video": rel[:-4], "path": str(cdf_root / rel),
                     "subset": rel.split("/")[0], "label": 1 - int(parts[0])})   # 1 - label_official
videos = pd.DataFrame(rows)
print("Celeb-DF root:", cdf_root)
print(videos.groupby(["subset", "label"]).size().to_string())
print("รวม", len(videos), "วิดีโอ (ควรเป็น 518)")

# ใช้ crop เดิมถ้ามี (จาก output ของระดับ 2)
prev_crops = sorted(INPUT_DIR.rglob("cdf_crops")) if INPUT_DIR.exists() else []
if prev_crops and not any(CROP_DIR.iterdir()):
    print("พบ cdf_crops เดิม:", prev_crops[0], "→ ใช้ซ้ำ")
    CROP_DIR = prev_crops[0]

In [ ]:
frame_meta_path = OUT_DIR / "frame_meta.csv"
if frame_meta_path.exists():
    frame_meta = pd.read_csv(frame_meta_path)
    print("ใช้ frame_meta เดิม:", len(frame_meta), "frame")
else:
    f_rows, t0 = [], time.time()
    for r in tqdm(videos.itertuples(index=False), total=len(videos), desc="videos"):
        out_dir = Path(CROP_DIR) / r.video.replace("/", "__")
        existing = sorted(out_dir.glob("*.png")) if out_dir.is_dir() else []
        if len(existing) >= N_FRAMES:
            recs = [{"crop_path": str(p), "frame_idx": int(p.stem), "shift": 0} for p in existing[:N_FRAMES]]
        else:
            try: recs, _ = process_video(r.path, out_dir)
            except Exception as e: recs = []
        for rec in recs:
            f_rows.append({"video": r.video, **rec})
    frame_meta = pd.DataFrame(f_rows)
    frame_meta.to_csv(frame_meta_path, index=False)
    print(f"crop เสร็จใน {(time.time()-t0)/60:.1f} นาที")

n_per_video = frame_meta.groupby("video").size()
print("frame ต่อวิดีโอ:", n_per_video.value_counts().sort_index().to_dict())
print("วิดีโอที่หาหน้าไม่เจอเลย:", len(videos) - n_per_video.index.nunique())

## 5. Inference → score ระดับวิดีโอ → AUC

In [ ]:
def to_tensor(rgb):
    x = (rgb.astype(np.float32) / 255.0 - np.array(MEAN, np.float32)) / np.array(STD, np.float32)
    return torch.from_numpy(x.transpose(2, 0, 1).copy())

class CropDataset(Dataset):
    def __init__(self, paths): self.paths = list(paths)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        rgb = np.asarray(Image.open(self.paths[i]).convert("RGB"))
        if rgb.shape[:2] != (IMG_SIZE, IMG_SIZE):
            rgb = cv2.resize(rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        return to_tensor(rgb)

@torch.no_grad()
def predict_batch(x):
    x = x.to(DEVICE, non_blocking=True)
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16,
                        enabled=USE_AMP and DEVICE.type == "cuda"):
        out = model(x)
    res = {}
    for name, logits in out.items():
        logits = logits.float()
        res[f"p_{name}"] = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        res[f"u_{name}"] = evidential_uncertainty(logits)[:, 0].cpu().numpy()
    return res

loader = DataLoader(CropDataset(frame_meta["crop_path"]), batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
t0 = time.time()
outs = [predict_batch(x) for x in tqdm(loader, desc="inference")]
pred = {k: np.concatenate([o[k] for o in outs]) for k in outs[0]}
frame_scores = pd.concat([frame_meta.reset_index(drop=True), pd.DataFrame(pred)], axis=1)
frame_scores = frame_scores.merge(videos[["video", "subset", "label"]], on="video")
frame_scores.to_csv(OUT_DIR / "frame_scores.csv", index=False)
print(f"inference {len(frame_scores)} frame ใน {time.time()-t0:.0f} s")

heads = ["main"] + (["fused", "branch2"] if DUAL else [])
agg = {f"p_{h}": "mean" for h in heads}; agg["u_main"] = "mean"; agg["frame_idx"] = "size"
vid = (frame_scores.groupby("video").agg(agg)
       .rename(columns={"frame_idx": "n_frames_used"}).reset_index())
vid = videos[["video", "subset", "label"]].merge(vid, on="video", how="left")
n_noface = int(vid["p_main"].isna().sum())
for h in heads: vid[f"p_{h}"] = vid[f"p_{h}"].fillna(0.5)
vid["u_main"] = vid["u_main"].fillna(vid["u_main"].mean())
vid.to_csv(OUT_DIR / "video_scores.csv", index=False)
print("วิดีโอที่ไม่มีหน้าเลย (ให้ score 0.5):", n_noface)

In [ ]:
def eer_score(y, s):
    fpr, tpr, _ = roc_curve(y, s); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr))
    return float((fpr[i] + fnr[i]) / 2)

def basic_metrics(y, s, thr=0.5):
    y, s = np.asarray(y).astype(int), np.asarray(s, dtype=float)
    pred = (s >= thr).astype(int)
    tnr = float(((pred == 0) & (y == 0)).sum() / max((y == 0).sum(), 1))
    tpr = float(((pred == 1) & (y == 1)).sum() / max((y == 1).sum(), 1))
    return {"n_real": int((y == 0).sum()), "n_fake": int((y == 1).sum()),
            "AUC": roc_auc_score(y, s), "AP": average_precision_score(y, s), "EER": eer_score(y, s),
            "ACC@0.5": float((pred == y).mean()), "BalACC@0.5": (tnr + tpr) / 2,
            "RealAcc(TNR)": tnr, "FakeAcc(TPR)": tpr}

def bootstrap_auc_ci(y, s, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); y, s = np.asarray(y), np.asarray(s)
    i0, i1 = np.where(y == 0)[0], np.where(y == 1)[0]
    aucs = [roc_auc_score(y[idx], s[idx]) for idx in
            (np.concatenate([rng.choice(i0, len(i0)), rng.choice(i1, len(i1))]) for _ in range(n_boot))]
    return np.percentile(aucs, [2.5, 97.5])

rows = []
for h in heads:
    m = basic_metrics(vid["label"], vid[f"p_{h}"])
    lo, hi = bootstrap_auc_ci(vid["label"].values, vid[f"p_{h}"].values)
    rows.append({"head": h, **m, "AUC_CI_low": lo, "AUC_CI_high": hi})
our_cdf = pd.DataFrame(rows)
our_cdf.to_csv(OUT_DIR / "metrics_celebdf.csv", index=False)

show = our_cdf.copy()
for c in ["AUC", "AP", "EER", "ACC@0.5", "BalACC@0.5", "RealAcc(TNR)", "FakeAcc(TPR)", "AUC_CI_low", "AUC_CI_high"]:
    show[c] = (100 * show[c]).round(2)
show["AUC [95% CI]"] = show.apply(lambda r: f"{r['AUC']:.2f} [{r['AUC_CI_low']:.2f}, {r['AUC_CI_high']:.2f}]", axis=1)
print("Celeb-DF v2 (518 วิดีโอ) — โมเดลที่เราเทรนเอง")
print(show[["head", "n_real", "n_fake", "AUC [95% CI]", "AP", "EER", "ACC@0.5",
            "RealAcc(TNR)", "FakeAcc(TPR)"]].to_string(index=False))
print(f"\npaper (Ours): Celeb-DF AUC = {PAPER['Celeb-DF']}")

## 6. เทียบ 3 ทาง: paper · checkpoint ผู้เขียน · โมเดลของเรา
`checkpoints/Ours/Ours/test/*.csv` ใน repo คือ score รายวิดีโอของผู้เขียน (เฉลี่ย 8 frame) —
คำนวณ AUC ใหม่เพื่อยืนยันว่าตัวเลขใน paper มาจากไฟล์นี้จริง แล้วเทียบ score รายคลิปกับของเรา

In [ ]:
author_rows, author_scores = [], {}
for name, auc_ref in PAPER.items():
    f = REPO_DIR / "checkpoints/Ours/Ours/test" / f"{name}.csv"
    if not f.exists(): continue
    d = pd.read_csv(f)
    author_scores[name] = d
    m = basic_metrics(d["label"], d["predict"])
    author_rows.append({"dataset": name, "n_videos": len(d), "AUC (จาก csv ผู้เขียน)": 100*m["AUC"],
                        "AUC (paper)": auc_ref, "ACC@0.5": 100*m["ACC@0.5"], "EER": 100*m["EER"]})
author_check = pd.DataFrame(author_rows)
print(author_check.round(2).to_string(index=False))

# --- ตารางสรุปเทียบ ---
ffpp_final = None
for p in (sorted(INPUT_DIR.rglob("cfb_run/final.json")) if INPUT_DIR.exists() else []):
    ffpp_final = json.load(open(p)); break
cmp_rows = [
    {"model": "paper — Ours (CFB)",        "FF++ c23": PAPER["FaceForensics_c23"], "Celeb-DF": PAPER["Celeb-DF"]},
    {"model": "paper — BEiT v2' (no CFB)", "FF++ c23": 99.14,                      "Celeb-DF": 89.18},
    {"model": "checkpoint ผู้เขียน (คำนวณใหม่)",
     "FF++ c23": 100*basic_metrics(author_scores["FaceForensics_c23"]["label"],
                                   author_scores["FaceForensics_c23"]["predict"])["AUC"]
                 if "FaceForensics_c23" in author_scores else np.nan,
     "Celeb-DF": 100*basic_metrics(author_scores["Celeb-DF"]["label"],
                                   author_scores["Celeb-DF"]["predict"])["AUC"]
                 if "Celeb-DF" in author_scores else np.nan},
    {"model": "เราเทรนเอง (ระดับ 3b)",
     "FF++ c23": 100*ffpp_final["test_auc"] if ffpp_final else np.nan,
     "Celeb-DF": 100*float(our_cdf.loc[our_cdf["head"] == "main", "AUC"].iloc[0])},
]
compare = pd.DataFrame(cmp_rows).round(2)
compare.to_csv(OUT_DIR / "comparison.csv", index=False)
print("\n" + compare.to_string(index=False))

In [ ]:
# --- เทียบ score รายคลิปกับของผู้เขียน ---
if "Celeb-DF" in author_scores:
    a = author_scores["Celeb-DF"].copy()
    key = [c for c in a.columns if a[c].dtype == object]
    if key:
        a["video"] = a[key[0]].astype(str).str.replace(".mp4", "", regex=False).str.replace("\\", "/", regex=False)
        mg = vid.merge(a[["video", "predict"]].rename(columns={"predict": "p_author"}), on="video", how="inner")
        print(f"\nจับคู่ได้ {len(mg)} / {len(vid)} คลิป")
        if len(mg) > 10:
            r = np.corrcoef(mg["p_main"], mg["p_author"])[0, 1]
            print(f"correlation ของ score รายคลิป (เรา vs ผู้เขียน): {r:.3f}")
            fig, ax = plt.subplots(figsize=(4.6, 4.4))
            for lab, col, nm in [(0, "#2a78d6", "real"), (1, "#eb6834", "fake")]:
                s = mg[mg["label"] == lab]
                ax.scatter(s["p_author"], s["p_main"], s=12, alpha=0.5, color=col, label=nm)
            ax.plot([0, 1], [0, 1], ls="--", lw=1, color="#52514e")
            ax.set(xlabel="P(fake) — checkpoint ผู้เขียน", ylabel="P(fake) — โมเดลของเรา",
                   title=f"Celeb-DF รายคลิป (r = {r:.3f})")
            ax.legend(frameon=False); plt.tight_layout()
            plt.savefig(OUT_DIR / "score_vs_author.png", dpi=150); plt.show()
    else:
        print("ไฟล์ csv ของผู้เขียนไม่มีคอลัมน์ชื่อวิดีโอ — ข้ามการเทียบรายคลิป")

### 6.1 ROC + การกระจายของ score

In [ ]:
C_REAL, C_FAKE, INK2, GRID = "#2a78d6", "#eb6834", "#52514e", "#e6e5e0"
plt.rcParams.update({"axes.edgecolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4))
for h, col in zip(heads, ["#2a78d6", "#1baf7a", "#8b5cf6"]):
    fpr, tpr, _ = roc_curve(vid["label"], vid[f"p_{h}"])
    axes[0].plot(fpr, tpr, lw=2, color=col,
                 label=f"{h}  (AUC {100*roc_auc_score(vid['label'], vid[f'p_{h}']):.2f})")
if "Celeb-DF" in author_scores:
    a = author_scores["Celeb-DF"]
    fpr, tpr, _ = roc_curve(a["label"], a["predict"])
    axes[0].plot(fpr, tpr, lw=1.6, ls=":", color="#eb6834",
                 label=f"ผู้เขียน (AUC {100*roc_auc_score(a['label'], a['predict']):.2f})")
axes[0].plot([0, 1], [0, 1], ls="--", lw=1, color=INK2)
axes[0].set(xlabel="False positive rate", ylabel="True positive rate", xlim=(0, 1), ylim=(0, 1.01),
            title="ROC — Celeb-DF v2 (video-level)")
axes[0].legend(loc="lower right", frameon=False)

bins = np.linspace(0, 1, 41)
for lab, col, nm in [(0, C_REAL, "real"), (1, C_FAKE, "fake")]:
    v = vid[vid["label"] == lab]["p_main"]
    axes[1].hist(v, bins=bins, density=True, alpha=0.25, color=col)
    axes[1].hist(v, bins=bins, density=True, histtype="step", lw=1.8, color=col, label=nm)
axes[1].axvline(0.5, color=INK2, ls="--", lw=1)
axes[1].set(xlabel="P(fake)", ylabel="density", title="การกระจายของ score (main branch)")
axes[1].legend(frameon=False)
for ax in axes: ax.grid(color=GRID, lw=0.8)
plt.tight_layout(); plt.savefig(OUT_DIR / "roc_celebdf.png", dpi=150); plt.show()

## 7. Threshold optimization (Sec. III-D, Eq. 10–11, Table IX)

- **τ = 0.5** — threshold มาตรฐาน
- **τ_ot** — เลือก threshold ที่ให้ ACC สูงสุดบน *source dataset* (ใช้ p ดิบ) แล้วนำไปใช้กับ *target*
- **τ_ot^UTOM** — เหมือน τ_ot แต่ปรับ p ก่อนด้วย `p' = tanh(λp / 2(u+ε))`, λ = 0.02, ε = 1e-7

source = FF++ c23 test (in-domain) · target = Celeb-DF (unseen) — ตรงกับสถานการณ์จริงที่ paper อธิบาย

In [ ]:
def utom_adjust(p, u, lam=UTOM_LAMBDA, eps=UTOM_EPS):
    return np.tanh(lam * p / (2 * (u + eps)))

def best_threshold(y, s):
    """Eq. 10-11: ลองทุกค่า score เป็น threshold แล้วเลือกตัวที่ ACC สูงสุด"""
    y, s = np.asarray(y).astype(int), np.asarray(s, dtype=float)
    o = np.argsort(s); ss, ys = s[o], y[o]
    real_below = np.concatenate([[0], np.cumsum(1 - ys)])[:-1]
    fake_above = ys.sum() - np.concatenate([[0], np.cumsum(ys)])[:-1]
    acc = (real_below + fake_above) / len(y)
    first = np.r_[True, ss[1:] != ss[:-1]]
    i = np.argmax(np.where(first, acc, -1))
    return float(ss[i]), float(acc[i])

acc_at = lambda y, s, t: float(((np.asarray(s) >= t).astype(int) == np.asarray(y)).mean())

# source = FF++ test จากระดับ 3b (ถ้ามี), ไม่งั้นใช้ครึ่งหนึ่งของ Celeb-DF
src = None
for p in (sorted(INPUT_DIR.rglob("cfb_run/test_scores.npy")) if INPUT_DIR.exists() else []):
    arr = np.load(p); src = {"name": "FF++ c23 test", "p": arr[0], "y": arr[1].astype(int)}; break

tgt = {"name": "Celeb-DF v2", "p": vid["p_main"].values, "y": vid["label"].values.astype(int),
       "u": vid["u_main"].values}
rows = []
if src is not None:
    t_ot, acc_src = best_threshold(src["y"], src["p"])
    rows.append({"source": src["name"], "target": tgt["name"], "threshold": "tau=0.5",
                 "tau": 0.5, "ACC_source": acc_at(src["y"], src["p"], 0.5),
                 "ACC_target": acc_at(tgt["y"], tgt["p"], 0.5)})
    rows.append({"source": src["name"], "target": tgt["name"], "threshold": "tau_ot",
                 "tau": t_ot, "ACC_source": acc_src, "ACC_target": acc_at(tgt["y"], tgt["p"], t_ot)})
    print("[NOTE] τ_ot^UTOM ต้องใช้ uncertainty ของ source ด้วย — ระดับ 3b ยังไม่ได้บันทึก u ของ FF++ test")
else:
    print("ไม่พบ test_scores.npy จากระดับ 3b → ใช้ Celeb-DF แบ่งครึ่งแทน (A = หา threshold, B = วัดผล)")
    rng = np.random.default_rng(SEED)
    perm = rng.permutation(len(vid)); A, B = perm[: len(perm)//2], perm[len(perm)//2:]
    pa, ya, ua = tgt["p"][A], tgt["y"][A], tgt["u"][A]
    pb, yb, ub = tgt["p"][B], tgt["y"][B], tgt["u"][B]
    t_ot, acc_a = best_threshold(ya, pa)
    t_ut, acc_au = best_threshold(ya, utom_adjust(pa, ua))
    rows += [
        {"source": "CDF ครึ่ง A", "target": "CDF ครึ่ง B", "threshold": "tau=0.5", "tau": 0.5,
         "ACC_source": acc_at(ya, pa, 0.5), "ACC_target": acc_at(yb, pb, 0.5)},
        {"source": "CDF ครึ่ง A", "target": "CDF ครึ่ง B", "threshold": "tau_ot", "tau": t_ot,
         "ACC_source": acc_a, "ACC_target": acc_at(yb, pb, t_ot)},
        {"source": "CDF ครึ่ง A", "target": "CDF ครึ่ง B", "threshold": "tau_ot^UTOM", "tau": t_ut,
         "ACC_source": acc_au, "ACC_target": acc_at(yb, utom_adjust(pb, ub), t_ut)},
    ]
thr = pd.DataFrame(rows)
for c in ["ACC_source", "ACC_target"]: thr[c] = (100 * thr[c]).round(2)
thr["tau"] = thr["tau"].round(4)
thr.to_csv(OUT_DIR / "threshold_analysis.csv", index=False)
print(thr.to_string(index=False))

## 8. สรุป

In [ ]:
our_main = 100 * float(our_cdf.loc[our_cdf["head"] == "main", "AUC"].iloc[0])
gap = our_main - PAPER["Celeb-DF"]
ffpp_line = (f"FF++ c23 test AUC     : {100*ffpp_final['test_auc']:.2f}   (paper {PAPER['FaceForensics_c23']})"
             if ffpp_final else "FF++ c23 test AUC     : (ไม่พบ final.json จากระดับ 3b)")

print(f"""
{'='*70}
สรุประดับ 3c — cross-dataset generalization
{'='*70}
checkpoint ที่ใช้       : {ckpt_file.name} (epoch {ck.get('epoch')})
เทรนบน                 : FF++ c23 เท่านั้น (ตาม paper)
ประเมินบน              : Celeb-DF v2 test 518 วิดีโอ (unseen dataset)

{ffpp_line}
Celeb-DF AUC (main)    : {our_main:.2f}   (paper {PAPER['Celeb-DF']})  ต่าง {gap:+.2f}
วิดีโอที่หาหน้าไม่เจอ    : {n_noface}
เวลารวม                : {(time.time()-T_START)/60:.1f} นาที
{'='*70}

ข้อควรระบุในรายงาน (ความต่างจาก implementation ของผู้เขียน)
  1. face tracking   ผู้เขียนใช้ FTCN ตรวจหน้าทุก frame แล้ว track หน้าที่อยู่นานที่สุด
                     notebook นี้ตรวจเฉพาะ 8 frame ที่เลือก + IoU tracking อย่างง่าย
  2. precision       เทรนด้วย AMP fp16 + gradient checkpointing (paper ใช้ fp32 บน 2xGTX3090)
  3. pre-training    ใช้ BEiT-1k-Face-55w.tar ของผู้เขียน ไม่ได้ pre-train เอง
                     (repo ไม่ได้แนบสคริปต์ SSL pre-training มา)
  4. LR schedule     cosine รายสเต็ป + warmup 1 epoch ตามค่าใน DBBF.yaml
""")
print("ไฟล์ผลลัพธ์:", *[p.name for p in sorted(OUT_DIR.iterdir())], sep="\n  ")